# Policies

> Every policy type a `World` can run, with the interface of stable-worldmodel.

In [ ]:
#| default_exp policy

In [ ]:
#| hide
from nbdev.showdoc import *

A policy receives the world's stacked **info dict** and returns one action per env and agent.

Every info value has a leading ``(num_envs, 1)`` (env, time), as in stable-worldmodel; per-agent
values add the agent axis after it, ``(num_envs, 1, num_agents, ...)``. Actions are returned
as an array of shape ``(num_envs, num_agents)``.

| policy | acts from |
|---|---|
| `RandomPolicy` | uniform samples of the action space |
| `ExpertPolicy` | the true state of each env (scripted, e.g. `GoToGoalPolicy`) |
| `FeedForwardPolicy` | one forward pass of a torch model (behaviour cloning, MARL actors) |
| `WorldModelPolicy` | plans with a solver over a world model (model-predictive control) |

In [ ]:
#| export
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Any, Callable

import numpy as np

from stable_marl.envs.base import MultiAgentEnv
from stable_marl.protocols import Actionable, Transformable

In [ ]:
#| export
class BasePolicy:
    """
    Base class of the policies a :class:`World` runs.

    Attributes
    ----------
    env : EnvPool
        The pool of envs the policy acts in (set by :meth:`set_env`)
    type : str
        Identifier of the policy type
    """
    env: Any
    type: str

    def __init__(self, **kwargs: Any):
        self.env = None
        self.type = 'base'
        for arg, value in kwargs.items():
            setattr(self, arg, value)

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        "Actions of shape (num_envs, num_agents) for the stacked `info_dict`."
        raise NotImplementedError

    def set_env(self, env: Any):
        "Attach the pool of envs this policy acts in."
        self.env = env

    def _prepare_info(self, info_dict: dict) -> dict:
        """
        A copy of `info_dict` ready for a torch model:

        * ``self.process[key]`` (a scaler with ``transform``, e.g. from sklearn) is applied to
          the features of `key`, i.e. its last axis
        * ``self.transform[key]`` (a callable on a batch of channel-first images) is applied to
          the images of `key`, of shape (..., H, W, C): it gets them as (N, C, H, W), and the
          result is reshaped back to (..., C', H', W')
        * numeric numpy arrays become torch tensors
        """
        import torch
        process, transform = getattr(self, 'process', {}), getattr(self, 'transform', {})
        out = {}
        for key, value in info_dict.items():
            if key in process:
                value = np.asarray(value)
                value = process[key].transform(value.reshape(-1, value.shape[-1])).reshape(value.shape)
            if key in transform:
                value = torch.as_tensor(value)
                images = value.reshape(-1, *value.shape[-3:]).permute(0, 3, 1, 2)
                images = transform[key](images)
                value = images.reshape(*value.shape[:-3], *images.shape[1:])
            if isinstance(value, np.ndarray) and value.dtype.kind not in 'USO':
                value = torch.from_numpy(value)
            out[key] = value
        return out

In [ ]:
#| export
class RandomPolicy(BasePolicy):
    "Samples uniformly random actions from the pool's batched action space."
    def __init__(self, seed: int | None = None, **kwargs: Any):
        """
        `seed` seeds the action space when the policy is attached to a pool,
        making the sampled actions reproducible.
        """
        super().__init__(**kwargs)
        self.type = 'random'
        self.seed = seed

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        return self.env.action_space.sample()

    def set_env(self, env: Any):
        super().set_env(env)
        if self.seed is not None:
            self._set_seed(self.seed)

    def _set_seed(self, seed: int):
        if self.env is not None:
            self.env.action_space.seed(seed)

In [ ]:
#| export
class ExpertPolicy(BasePolicy):
    """
    Base class of privileged (scripted) policies, which read the true state of each env
    instead of the info dict, e.g. a shortest-path planner to collect expert datasets.

    Subclasses implement :meth:`act`, the actions of all the agents of one env.
    """
    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)
        self.type = 'expert'

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        return np.array([self.act(env.unwrapped) for env in self.env.envs])

    def act(self, env: MultiAgentEnv) -> np.ndarray:
        "Actions of the agents of `env`, of shape (num_agents,)."
        raise NotImplementedError

In [ ]:
#| export
class FeedForwardPolicy(BasePolicy):
    """
    Computes actions with one forward pass of a torch model, e.g. a behaviour-cloned policy or
    a trained MARL actor.

    ``model.get_action(info_dict)`` receives the prepared info dict (see
    :meth:`BasePolicy._prepare_info`), with its tensors on the model's device, and returns
    actions of shape (num_envs, num_agents). ``process['action']``, if given, maps them back
    with its ``inverse_transform``.
    """
    def __init__(
        self,
        model: Actionable,
        process: dict[str, Transformable] | None = None,
        transform: dict[str, Callable] | None = None,
        **kwargs: Any):
        if not isinstance(model, Actionable):
            raise TypeError(f"FeedForwardPolicy needs a model with `get_action` (Actionable), got {type(model).__name__}")
        super().__init__(**kwargs)
        self.type = 'feed_forward'
        self.model = model.eval()
        self.process = process or {}
        self.transform = transform or {}

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        import torch
        info = self._prepare_info(info_dict)
        parameter = next(self.model.parameters(), None)
        if parameter is not None:
            info = {k: v.to(parameter.device) if torch.is_tensor(v) else v for k, v in info.items()}
        with torch.no_grad():
            action = self.model.get_action(info)
        if torch.is_tensor(action):
            action = action.cpu().numpy()
        if 'action' in self.process:
            action = self.process['action'].inverse_transform(action)
        return action

In [ ]:
#| export
@dataclass(frozen=True)
class PlanConfig:
    """
    Configuration of the model-predictive control loop of a :class:`WorldModelPolicy`
    (same fields as stable-worldmodel).

    Attributes
    ----------
    horizon : int
        Planning horizon, in steps
    receding_horizon : int
        Steps of each plan executed before replanning
    history_len : int
        Context frames given to the world model (one every ``action_block`` env steps)
    history_max_len : int, optional
        Capacity of the context buffer
    action_block : int
        Env steps per planning step (each planned action covers ``action_block`` env steps)
    warm_start : bool
        Whether the rest of the previous plan is passed to the solver
    """
    horizon: int
    receding_horizon: int
    history_len: int = 1
    history_max_len: int | None = None
    action_block: int = 1
    warm_start: bool = True

    def __post_init__(self):
        if self.history_len < 1:
            raise ValueError(f"history_len must be >= 1, got {self.history_len}")
        if not 1 <= self.receding_horizon <= self.horizon:
            raise ValueError(f"receding_horizon must be in [1, horizon], got {self.receding_horizon}")

    @property
    def plan_len(self) -> int:
        "Length of a plan, in env steps."
        return self.horizon * self.action_block

In [ ]:
#| export
ACTION_HISTORY_KEY = 'action_history'   # info key of the actions executed between the context frames


class WorldModelPolicy(BasePolicy):
    """
    Model-predictive control: plans the actions of all agents with a solver (over a world model),
    executes the first ``receding_horizon`` planning steps (each ``action_block`` env steps), then
    replans. Each env replans on its own schedule, and from scratch when it starts an episode
    (``_needs_flush``). Works with discrete and continuous actions; centralised or decentralised
    planning is chosen by the solver's ``mode`` and the objective (joint or per-agent cost).
    Continuous actions are clipped to the action space, so the recorded actions are the executed ones.

    With ``history_len > 1``, the solver gets the last ``history_len`` frames of the
    `history_keys` (one every ``action_block`` env steps, oldest first, e.g. ``pov``
    ``(n, history_len, num_agents, ...)``) and the actions executed between them under
    ``'action_history'`` ``(n, history_len - 1, num_agents, action_block * D)``. Early in an
    episode the context is shorter; it is padded (copies of the oldest frame, zero actions) only
    when envs at different fill levels replan together.
    """
    def __init__(
        self,
        solver: Any,
        config: PlanConfig,
        process: dict[str, Transformable] | None = None,
        transform: dict[str, Callable] | None = None,
        history_keys: tuple[str, ...] = ('pov',),
        **kwargs: Any):
        super().__init__(**kwargs)
        self.type = 'world_model'
        self.cfg, self.solver = config, solver
        if hasattr(getattr(solver, 'cost', None), 'eval'):   # plan with the model in inference mode (e.g. BatchNorm)
            solver.cost.eval()
        self.process, self.transform = process or {}, transform or {}
        self.history_keys = tuple(history_keys)
        self._action_buffer: list[deque] | None = None
        self._next_init = None
        self._history_buffer = None

    @property
    def flatten_receding_horizon(self) -> int:
        "Executed env steps of a plan."
        return self.cfg.receding_horizon * self.cfg.action_block

    def set_env(self, env: Any):
        from stable_marl.buffer import HistoryBuffer
        from stable_marl.planning.solver import Solver
        self.env = env
        self.solver.configure(action_space=env.single_action_space, n_envs=env.num_envs, config=self.cfg)
        if not isinstance(self.solver, Solver):   # checked once configured, as its properties need the config
            raise TypeError("The solver must implement the Solver protocol")
        self._action_buffer = [deque(maxlen=self.flatten_receding_horizon) for _ in range(env.num_envs)]
        self._next_init = None
        self._history_buffer = None
        if self.cfg.history_len > 1:
            max_len = self.cfg.history_max_len or (self.cfg.history_len - 1) * self.cfg.action_block + 1
            self._history_buffer = HistoryBuffer(env.num_envs, max_len, self.cfg.action_block, block_keys=('action',))

    @property
    def _discrete(self) -> bool:
        return 'Discrete' in type(self.env.single_action_space).__name__

    def _env_steps(self, plan):
        """
        Planned actions ``(n, rh, A, X)`` -> one action per env step ``(n, rh * action_block, A, *shape)``:
        X is ``action_block`` indices (discrete) or ``action_block * D`` values (continuous).
        """
        n, rh, A = plan.shape[:3]
        ab = self.cfg.action_block
        if self._discrete:
            return plan.reshape(n, rh, A, ab).transpose(2, 3).reshape(n, rh * ab, A)
        shape = tuple(self.env.single_action_space.shape[1:])
        return plan.reshape(n, rh, A, ab, -1).transpose(2, 3).reshape(n, rh * ab, A, *shape)

    def get_action(self, info_dict: dict, **kwargs: Any) -> np.ndarray:
        import torch
        n_envs = self.env.num_envs
        needs_flush = info_dict.pop('_needs_flush', None)
        if needs_flush is not None:
            flush = [int(i) for i in np.flatnonzero(needs_flush)]
            for i in flush:
                self._action_buffer[i].clear()
                if self._next_init is not None:
                    self._next_init[i] = 0
            if self._history_buffer is not None and flush:
                self._history_buffer.reset(flush)
        info = self._prepare_info(info_dict)
        if self._history_buffer is not None:
            self._history_buffer.append({k: info[k] for k in (*self.history_keys, 'action') if k in info})

        terminated = info.get('terminated')
        dead = (np.asarray(terminated).reshape(n_envs, -1).all(axis=1) if terminated is not None
                else np.zeros(n_envs, bool))
        replan = [i for i in range(n_envs) if not self._action_buffer[i] and not dead[i]]
        if replan:
            idx = torch.as_tensor(replan, dtype=torch.long)
            sliced = {k: v[idx] if torch.is_tensor(v) else v[replan] if isinstance(v, np.ndarray)
                      else [v[i] for i in replan] if isinstance(v, list) else v
                      for k, v in info.items()}
            if self._history_buffer is not None:
                n_frames = min(self.cfg.history_len, max(self._history_buffer.num_strided(replan)))
                history = self._history_buffer.get(n_frames, env_ids=replan)
                for k in self.history_keys:
                    if k in history:
                        sliced[k] = history[k]
                if 'action' in history:
                    sliced[ACTION_HISTORY_KEY] = history['action']
            init = self._next_init[idx] if self._next_init is not None else None
            actions = self.solver(sliced, init_action=init)['actions']          # (n, horizon, A, X)
            keep = self.cfg.receding_horizon
            plan, rest = actions[:, :keep], actions[:, keep:]
            if self.cfg.warm_start and rest.shape[1] > 0:
                if self._next_init is None:
                    self._next_init = torch.zeros(n_envs, *rest.shape[1:], dtype=rest.dtype)
                self._next_init[idx] = rest
            elif not self.cfg.warm_start:
                self._next_init = None
            steps = self._env_steps(plan)
            for row, i in enumerate(replan):
                self._action_buffer[i].extend(steps[row])

        shape = (n_envs, *self.env.single_action_space.shape)
        action = np.zeros(shape, np.int64) if self._discrete else np.full(shape, np.nan, np.float32)
        for i in range(n_envs):
            if not dead[i]:
                action[i] = np.asarray(self._action_buffer[i].popleft())
        if 'action' in self.process:
            action = self.process['action'].inverse_transform(action)
        space = self.env.single_action_space
        if not self._discrete and hasattr(space, 'low'):   # solvers may plan beyond the bounds: execute (and record) valid actions
            action = np.clip(action, space.low, space.high)
        return action


Policy = BasePolicy   # alias, as in stable-worldmodel

Tested with the `World` in `world/01_world.ipynb`.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()